# Data Science Programming - Sesion 4: Limpieza y transformacion de datos

**Especializacion en Ciencia de Datos - Universidad Santo Tomas - Tunja**
**Prof. Hugo Hernandez | Martes 15 de septiembre, 2026**

---

## Que haremos hoy

1. **Diagnosticar**: mirar la tabla antes de tocarla
2. **Valores faltantes**: detectarlos e imputarlos con media, mediana y KNN
3. **Duplicados**: encontrarlos y quitarlos
4. **Texto inconsistente y valores fuera de rango**
5. **Outliers**: Z-Score e IQR
6. **Escalamiento**: MinMaxScaler y StandardScaler
7. **Funciones reutilizables**: `pipeline_limpieza(df)`
8. **Formatos**: CSV, JSON y HDF5

> Esta es la sesion que sostiene la **Actividad 2 (25%)**.

Ejecuta las celdas en orden con `Shift + Enter`.

---
## Antes de empezar

Los datos reales llegan sucios: huecos, filas repetidas, texto escrito de tres formas,
valores absurdos. Entre el 60% y el 80% del tiempo de un analista se va en limpiar.

### El orden importa

| Paso | Que hacemos |
|---|---|
| 1 | Diagnosticar |
| 2 | Duplicados |
| 3 | Texto y rangos |
| 4 | Outliers |
| 5 | Faltantes |
| 6 | Escalar |

Hoy usamos `pandas`, `numpy` y `scikit-learn`. Instalalas con la celda de abajo si aun no las tienes.

In [ ]:
%pip install pandas numpy scikit-learn matplotlib --quiet
print("Librerias listas")

---
## Parte 1 - La tabla sucia

Ventas de una tienda de barrio. La construimos a mano para saber exactamente que defectos
tiene: una fila repetida, 3 faltantes en `precio`, un outlier, texto inconsistente y una
calificacion fuera de rango.

In [ ]:
import pandas as pd
import numpy as np

ventas = pd.DataFrame({
    'id_venta': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110,
                 111, 112, 113, 114, 115, 116, 117, 118, 119, 105],
    'vendedor': ['Ana', 'luis', 'MARTA', ' Ana ', 'Sofia', 'ana', 'Luis ', 'marta', 'SOFIA', 'Ana',
                 'luis', 'Marta', 'sofia', 'ANA', 'Luis', 'marta ', 'Sofia', 'ana', 'Luis', 'Sofia'],
    'ciudad':   ['Tunja', 'bogota', 'TUNJA', 'Tunja ', 'Duitama', 'tunja', 'Bogota', 'duitama',
                 'Duitama', 'Tunja', 'BOGOTA', 'Tunja', 'duitama', 'Tunja ', 'bogota', 'Tunja',
                 'Duitama', 'tunja', 'Bogota', 'Duitama'],
    'unidades': [3, 5, 2, 4, 1, 6, 3, 2, 4, 5, 2, 3, 1, 4, 6, 2, 3, 5, 4, 1],
    'precio':   [12000.0, 18000.0, np.nan, 15000.0, 9000.0, 21000.0, 980000.0, np.nan, 16000.0,
                 13000.0, 17000.0, 14000.0, 10000.0, 15000.0, 19000.0, 11000.0, 20000.0, np.nan,
                 18000.0, 9000.0],
    'calificacion': [5, 4, 3, 5, 4, 9, 2, 3, 4, 5, 4, 3, 5, 4, 2, 3, 5, 4, 4, 4],
})

ventas

### 1.1 Guardarla y volverla a cargar como CSV

- `df.to_csv('archivo.csv', index=False)` -> guardar
- `pd.read_csv('archivo.csv')` -> leer

In [ ]:
ventas.to_csv('ventas_sucias.csv', index=False)
ventas = pd.read_csv('ventas_sucias.csv')

print('Filas y columnas:', ventas.shape)
print(ventas.head(3))

---
## Parte 2 - Diagnostico: mirar antes de tocar

| Pregunta | Comando |
|---|---|
| Que tamano y tipos? | `df.shape`, `df.dtypes`, `df.info()` |
| Cuantos huecos? | `df.isna().sum()` |
| Filas repetidas? | `df.duplicated().sum()` |
| Los numeros tienen sentido? | `df.describe()` |

In [ ]:
print('--- Forma (filas, columnas) ---')
print(ventas.shape)

print('\n--- Tipo de dato de cada columna ---')
print(ventas.dtypes)

print('\n--- Huecos por columna ---')
print(ventas.isna().sum())

print('\n--- Filas duplicadas ---')
print(ventas.duplicated().sum())

In [ ]:
ventas.describe()

### 2.1 Que nos dijo la radiografia

- `count` de `precio` es 17, no 20: faltan 3 datos.
- `mean` 71588 contra mediana 15000: hay un valor gigante jalando.
- `max` de `calificacion` es 9 en una escala de 1 a 5.
- Hay 1 fila duplicada.

### 2.2 Cuidado: no todo lo que parece numero se puede promediar

In [ ]:
print('Media de precio:', ventas['precio'].mean())

print('\nCuantas ventas por vendedor:')
print(ventas['vendedor'].value_counts())

print('\nMedia de vendedor:', ventas['vendedor'].mean())

---
## Parte 3 - Valores faltantes

Un hueco (`NaN`) **no es un cero**: el cero es un valor, el hueco es la ausencia de valor.

| Estrategia | Comando | Cuando conviene |
|---|---|---|
| **Eliminar** la fila | `df.dropna()` | pocos huecos y muchas filas |
| **Imputar** (rellenar) | `df['col'].fillna(valor)` | cuando perder la fila cuesta caro |

In [ ]:
print('Huecos por columna:')
print(ventas.isna().sum())

porcentaje = ventas['precio'].isna().sum() / len(ventas) * 100
print(f'\nEn precio falta el {porcentaje:.1f}% de los datos')

print('\nFilas donde falta el precio:')
print(ventas[ventas['precio'].isna()])

### 3.1 Media o mediana?

- **Media**: suma todo y divide. Cada valor empuja en proporcion a su tamano.
- **Mediana**: ordena y toma el del medio. Solo importa la posicion.

Si en un bar entra un millonario, el sueldo **promedio** se dispara pero el **mediano**
casi no se mueve.

| Situacion | Usa |
|---|---|
| Datos simetricos, sin extremos | media |
| Hay outliers | **mediana** |
| Texto / categorias | moda: `df['col'].mode()[0]` |

In [ ]:
con_media = ventas.copy()
con_mediana = ventas.copy()

media = ventas['precio'].mean()
mediana = ventas['precio'].median()

print(f'Media   : {media:>12,.0f}')
print(f'Mediana : {mediana:>12,.0f}')

con_media['precio'] = con_media['precio'].fillna(media)
con_mediana['precio'] = con_mediana['precio'].fillna(mediana)

print('\nCon que quedo rellenada la fila 2:')
print('  imputando con media  :', round(con_media.loc[2, 'precio']))
print('  imputando con mediana:', round(con_mediana.loc[2, 'precio']))

### 3.2 El nombre de la columna es literal

In [ ]:
print(ventas.columns.tolist())

prueba = ventas.copy()
prueba['precio'] = prueba['precio'].fillna(0)
print('Huecos despues de rellenar con 0:', prueba['precio'].isna().sum())

prueba['precios'] = prueba['precios'].fillna(0)

### 3.3 KNN Imputation: preguntarle a los vecinos

Para rellenar el hueco de una fila, busca las **K filas mas parecidas** en las demas
columnas numericas y promedia el valor que ellas tienen.

```python
from sklearn.impute import KNNImputer
imputador = KNNImputer(n_neighbors=3)
resultado = imputador.fit_transform(tabla_numerica)
```

Ojo: solo acepta numeros y devuelve un array de NumPy, no un DataFrame.

In [ ]:
from sklearn.impute import KNNImputer

imputador = KNNImputer(n_neighbors=3)
completas = imputador.fit_transform(ventas)

ventas_knn = pd.DataFrame(completas, columns=ventas.columns)

print('Precios que KNN invento para las filas 2, 7 y 17:')
print(ventas_knn.loc[[2, 7, 17], 'precio'])

### 3.4 Cual de las tres usar en la Actividad 2

| Metodo | Ventaja | Desventaja |
|---|---|---|
| Media | simple, rapida | se la lleva el outlier |
| Mediana | resiste outliers | ignora el resto de la fila |
| KNN | usa el contexto de la fila | mas lento, exige escalar y solo numeros |

Lo que se califica es la **justificacion escrita**, no la linea de codigo.

---
## Parte 4 - Duplicados

- `df.duplicated()` -> `True` en cada fila que **ya habia aparecido antes**.
- `df.drop_duplicates()` -> la tabla sin las repetidas.

`keep='first'` (por defecto), `keep='last'`, `keep=False` (bota todas las copias).
`subset=['id_venta']` define duplicado por columnas concretas.

In [ ]:
print('Filas duplicadas:', ventas.duplicated().sum())

print('\nLas filas implicadas (original y copia):')
print(ventas[ventas.duplicated(keep=False)])

### 4.1 Quitarlas

`drop_duplicates()` **no modifica** la tabla: devuelve una tabla nueva. Hay que reasignar:

```python
ventas = ventas.drop_duplicates()
```

In [ ]:
antes = len(ventas)

sin_duplicados = ventas.drop_duplicates(keep='ultimo').reset_index(drop=True)

print(f'Filas: {antes} -> {len(sin_duplicados)}')
print('Duplicados que quedan:', sin_duplicados.duplicated().sum())

---
## Parte 5 - Texto inconsistente y valores fuera de rango

Para el computador `Tunja`, `tunja`, `TUNJA` y ` Tunja ` son **cuatro ciudades distintas**.

| Comando | Que hace |
|---|---|
| `.str.strip()` | quita espacios de los extremos |
| `.str.lower()` | todo a minusculas |
| `.str.title()` | primera letra en mayuscula |

In [ ]:
limpia = ventas.copy()

print('ANTES  ciudad  :', sorted(limpia['ciudad'].unique().tolist()))
print('ANTES  vendedor:', sorted(limpia['vendedor'].unique().tolist()))

limpia['ciudad'] = limpia['ciudad'].str.strip().str.title()
limpia['vendedor'] = limpia['vendedor'].str.strip().str.title()

print('\nDESPUES ciudad  :', sorted(limpia['ciudad'].unique().tolist()))
print('DESPUES vendedor:', sorted(limpia['vendedor'].unique().tolist()))

### 5.1 Por que esto no es cosmetica

In [ ]:
print('Ventas por ciudad ANTES de limpiar el texto:')
print(ventas.groupby('ciudad').size())

print('\nVentas por ciudad DESPUES de limpiar el texto:')
print(limpia.groupby('ciudad').size())

### 5.2 Valores fuera de rango

Un dato que viola una regla del negocio: edad de 200, calificacion de 9 en escala de 1 a 5.

| Estrategia | Comando |
|---|---|
| Recortar al limite | `df['col'].clip(1, 5)` |
| Convertirlo en hueco | poner `np.nan` y despues imputar |

In [ ]:
fuera = limpia[(limpia['calificacion'] < 1) | (limpia['calificacion'] > 5)]
print('Filas con calificacion fuera de rango:', len(fuera))
print(fuera[['id_venta', 'vendedor', 'calificacion']])

recortada = limpia['calificacion'].clip(1, 5)
print('\nOpcion A - recortar al limite:', recortada.tolist())

limpia.loc[limpia['calificacion'] > 5, 'calificacion'] = np.nan
print('\nOpcion B - volverlo hueco. Huecos en calificacion:', limpia['calificacion'].isna().sum())

limpia['calificacion'] = limpia['calificacion'].fillna(limpia['calificacion'].median())
print('Ya imputado. Maximo de calificacion:', limpia['calificacion'].max())

---
## Parte 6 - Outliers: Z-Score e IQR

Un **outlier** es un dato que se sale del patron del resto. No siempre es un error, pero
mientras no lo mires no sabes si es un cliente especial o un dedo que resbalo en el teclado.

La rubrica de la Actividad 2 nombra los dos metodos: Z-Score e IQR.

### 6.1 Metodo 1: Z-Score

$$z = \frac{x - \text{media}}{\text{desviacion estandar}}$$

Criterio: **`|z| > 3`** es outlier. Sale de que en una campana de Gauss el 99,7% de los
datos cae dentro de tres desviaciones.

Ojo: el propio outlier infla la media y la desviacion, asi que con pocos datos puede
esconderse a si mismo (*enmascaramiento*).

In [ ]:
columna = limpia['precio']

media_p = columna.mean()
desviacion = columna.std()

print(f'Media       : {media_p:>12,.0f}')
print(f'Desviacion  : {desviacion:>12,.0f}')

z = (columna - media_p) / desviacion
limpia['z_precio'] = z.abs()

print('\nZ-Score de cada precio (ordenado de mayor a menor):')
print(limpia[['id_venta', 'precio', 'z_precio']].sort_values('z_precio', ascending=False).head(4))

outliers_z = limpia[limpia['z_precio'] > 3]
print(f'\nOutliers segun Z-Score (|z| > 3): {len(outliers_z)}')
print(outliers_z[['id_venta', 'precio', 'z_precio']])

### 6.2 Metodo 2: IQR (rango intercuartil)

```
  |----25%----|----25%----|----25%----|----25%----|
  min         Q1        mediana       Q3        max
```

`IQR = Q3 - Q1`

| Limite | Formula |
|---|---|
| Inferior | `Q1 - 1.5 * IQR` |
| Superior | `Q3 + 1.5 * IQR` |

Es lo que dibuja un boxplot: la caja va de Q1 a Q3 y los puntos fuera de los bigotes son
los outliers. En Pandas: `df['col'].quantile(0.25)` (0.25, no 25).

In [ ]:
q1 = limpia['precio'].quantile(0.25)
q3 = limpia['precio'].quantile(0.75)
iqr = q3 - q1

limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

print(f'Q1              : {q1:>10,.0f}')
print(f'Q3              : {q3:>10,.0f}')
print(f'IQR (Q3 - Q1)   : {iqr:>10,.0f}')
print(f'Limite inferior : {limite_inferior:>10,.0f}')
print(f'Limite superior : {limite_superior:>10,.0f}')

es_outlier = (limpia['precio'] < limite_inferior) | (limpia['precio'] > limite_superior)

print(f'\nOutliers segun IQR: {es_outlier.sum()}')
print(limpia.loc[es_outlier, ['id_venta', 'vendedor', 'precio']])

### 6.3 Los dos metodos, cara a cara

| | Z-Score | IQR |
|---|---|---|
| En que se basa | media y desviacion | cuartiles (posiciones) |
| Criterio | `abs(z) > 3` | fuera de `Q1-1.5*IQR` y `Q3+1.5*IQR` |
| Lo afecta el outlier? | **si** | **no** |
| Supone campana de Gauss? | si | no |

Que hacer con el outlier: investigarlo, eliminar la fila o recortarlo al limite
(*winsorizacion*). Lo que no se vale es borrarlo sin decirlo en el informe.

In [ ]:
sin_outliers = limpia[~es_outlier].copy()

print(f'Filas: {len(limpia)} -> {len(sin_outliers)}')

print('\nEfecto sobre las estadisticas de precio:')
print(f'  Media    ANTES: {limpia["precio"].mean():>10,.0f}   DESPUES: {sin_outliers["precio"].mean():>10,.0f}')
print(f'  Maximo   ANTES: {limpia["precio"].max():>10,.0f}   DESPUES: {sin_outliers["precio"].max():>10,.0f}')
print(f'  Mediana  ANTES: {limpia["precio"].median():>10,.0f}   DESPUES: {sin_outliers["precio"].median():>10,.0f}')

---
## Parte 7 - Escalamiento: poner todo en la misma regla

| | MinMaxScaler | StandardScaler |
|---|---|---|
| Que hace | aplasta todo al rango **0 a 1** | centra en **media 0, desviacion 1** |
| Formula | `(x - min) / (max - min)` | `(x - media) / desviacion` |
| Le afectan los outliers | mucho | menos, pero si |
| Usalo cuando | necesitas un rango fijo | el metodo asume datos centrados |

**StandardScaler es exactamente el Z-Score de la Parte 6.** Misma formula, otro uso.

### 7.1 MinMaxScaler y la regla de los dos corchetes

```python
escalador = MinMaxScaler()
resultado = escalador.fit_transform(X)
```

- `df['precio']` es una **Series** (una dimension). No sirve.
- `df[['precio']]` es un **DataFrame** de una columna (dos dimensiones). Ese si.

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

base = sin_outliers.drop(columns='z_precio').copy()
base['precio'] = base['precio'].fillna(base['precio'].median())

escalador = MinMaxScaler()
base['precio_minmax'] = escalador.fit_transform(base['precio'])

print(base[['id_venta', 'precio', 'precio_minmax']].head(6))
print(f'\nMinimo escalado: {base["precio_minmax"].min():.2f}  (siempre 0)')
print(f'Maximo escalado: {base["precio_minmax"].max():.2f}  (siempre 1)')

### 7.2 StandardScaler y los dos resultados lado a lado

In [ ]:
comparacion = pd.DataFrame({'precio': base['precio']})

comparacion['minmax'] = MinMaxScaler().fit_transform(base[['precio']])
comparacion['estandar'] = StandardScaler().fit_transform(base[['precio']])

print(comparacion.sort_values('precio').round(3).to_string(index=False))

print('\nResumen de cada escala:')
print(f'  minmax   -> min {comparacion["minmax"].min():.2f}  max {comparacion["minmax"].max():.2f}  media {comparacion["minmax"].mean():.2f}')
print(f'  estandar -> min {comparacion["estandar"].min():.2f}  max {comparacion["estandar"].max():.2f}  media {comparacion["estandar"].mean():.2f}')

### 7.3 Dos advertencias que valen puntos

**1. Los huecos no dan error: se cuelan.** Los escaladores aceptan `NaN` y devuelven `NaN`
en esa posicion, sin avisar. Imputa primero, escala despues.

**2. Escalar no siempre hace falta:**

| Metodo posterior | Necesita escalar? |
|---|---|
| KNN, K-Means, SVM, PCA | **si**, miden distancias |
| Redes neuronales, regresion regularizada | **si** |
| Arboles de decision, Random Forest | **no** |
| Solo describir y graficar | **no** |

In [ ]:
mini = pd.DataFrame({'valor': [100.0, np.nan, 300.0]})

resultado = MinMaxScaler().fit_transform(mini[['valor']])

print('Entrada :', mini['valor'].tolist())
print('Salida  :', resultado.ravel().tolist())

---
## Parte 8 - Funciones reutilizables

La rubrica de la Actividad 2 lo pide con estas palabras:

> *"Creacion de funciones reutilizables para cada paso del proceso de limpieza y
> transformacion."*

### Las tres reglas de una buena funcion de limpieza

1. **Recibe un DataFrame y devuelve un DataFrame** (para poder encadenarlas).
2. **No modifica la tabla que le entra**: trabaja sobre una `.copy()`.
3. **Hace una sola cosa** y su nombre lo dice.

In [ ]:
def limpiar_duplicados(df):
    """Quita filas repetidas y renumera el indice. Devuelve una tabla nueva."""
    return df.drop_duplicates(keep='first').reset_index(drop=True)


def limpiar_texto(df, columnas):
    """Quita espacios sobrantes y unifica mayusculas en las columnas indicadas."""
    salida = df.copy()
    for col in columnas:
        salida[col] = salida[col].str.strip().str.title()
    return salida


def corregir_fuera_de_rango(df, col, minimo, maximo):
    """Convierte en hueco (NaN) todo valor de col que se salga de [minimo, maximo]."""
    salida = df.copy()
    fuera = (salida[col] < minimo) | (salida[col] > maximo)
    salida.loc[fuera, col] = np.nan
    return salida


prueba1 = limpiar_duplicados(ventas)
print('limpiar_duplicados      :', len(ventas), '->', len(prueba1), 'filas')

prueba2 = limpiar_texto(prueba1, ['vendedor', 'ciudad'])
print('limpiar_texto           :', sorted(prueba2['ciudad'].unique().tolist()))

prueba3 = corregir_fuera_de_rango(prueba2, 'calificacion', 1, 5)
print('corregir_fuera_de_rango : huecos nuevos en calificacion =',
      prueba3['calificacion'].isna().sum())

### 8.1 Funciones con opciones: parametros por defecto

`factor=1.5` significa "si no me dices nada, uso 1.5". El `raise ValueError` hace que la
funcion avise en voz alta cuando le piden algo que no sabe hacer.

In [ ]:
def quitar_outliers_iqr(df, col, factor=1.5):
    """Elimina las filas cuyo valor de col cae fuera de los limites de Tukey."""
    salida = df.copy()
    q1 = salida[col].quantile(0.25)
    q3 = salida[col].quantile(0.75)
    iqr = q3 - q1
    inferior = q1 - factor * iqr
    superior = q3 + factor * iqr
    dentro = (salida[col] >= inferior) | (salida[col].isna())
    dentro = dentro & ((salida[col] <= superior) | (salida[col].isna()))
    return salida[dentro].reset_index(drop=True)


def imputar_faltantes(df, col, estrategia='mediana'):
    """Rellena los huecos de col con media, mediana o moda."""
    salida = df.copy()
    if estrategia == 'media':
        valor = salida[col].mean()
    elif estrategia == 'mediana':
        valor = salida[col].median()
    elif estrategia == 'moda':
        valor = salida[col].mode()[0]
    else:
        raise ValueError("estrategia debe ser 'media', 'mediana' o 'moda'")
    salida[col] = salida[col].fillna(valor)
    return salida


def escalar(df, col, metodo='minmax'):
    """Agrega una columna col_minmax o col_estandar con la variable escalada."""
    salida = df.copy()
    if metodo == 'minmax':
        escalador = MinMaxScaler()
    elif metodo == 'estandar':
        escalador = StandardScaler()
    else:
        raise ValueError("metodo debe ser 'minmax' o 'estandar'")
    salida[col + '_' + metodo] = escalador.fit_transform(salida[[col]])
    return salida


prueba4 = quitar_outliers_iqr(prueba3, 'precio')
print('quitar_outliers_iqr :', len(prueba3), '->', len(prueba4), 'filas')

prueba5 = imputar_faltantes(prueba4, 'precio', 'mediana')
print('imputar_faltantes   : huecos en precio =', prueba5['precio'].isna().sum())

prueba6 = escalar(prueba5, 'precio', 'minmax')
print('escalar             : columnas =', prueba6.columns.tolist())

### 8.2 El pipeline: una sola funcion que las encadena

Orden dentro del pipeline:

1. duplicados
2. texto
3. fuera de rango
4. outliers
5. faltantes
6. escalar

El dia que llegue el archivo del mes siguiente, limpiarlo es **una linea**.

In [ ]:
def pipeline_limpieza(df, verbose=True):
    """Aplica todos los pasos de limpieza en orden y devuelve la tabla lista."""
    filas_iniciales = len(df)

    salida = limpiar_duplicados(df)
    if verbose:
        print(f'1. Duplicados     : {filas_iniciales} -> {len(salida)} filas')

    salida = limpiar_texto(salida, ['vendedor', 'ciudad'])
    if verbose:
        print(f'2. Texto          : {salida["ciudad"].nunique()} ciudades reales')

    salida = corregir_fuera_de_rango(salida, 'calificacion', 1, 5)
    if verbose:
        print(f'3. Fuera de rango : {salida["calificacion"].isna().sum()} valores anulados')

    antes_outliers = len(salida)
    salida = quitar_outliers_iqr(salida, 'precio')
    if verbose:
        print(f'4. Outliers       : {antes_outliers} -> {len(salida)} filas')

    salida = imputar_faltantes(salida, 'precio', 'mediana')
    salida = imputar_faltantes(salida, 'calificacion', 'mediana')
    if verbose:
        print(f'5. Faltantes      : {salida.isna().sum().sum()} huecos restantes')

    salida = escalar(salida, 'precio', 'minmax')
    if verbose:
        print(f'6. Escalado       : columna precio_minmax agregada')

    return salida


ventas_limpias = pipeline_limpieza(ventas)

print('\nTabla final:')
ventas_limpias

### 8.3 La auditoria final

In [ ]:
print('COMPROBACION FINAL')
print('-' * 42)
print('Filas                    :', len(ventas), '->', len(ventas_limpias))
print('Huecos totales           :', ventas.isna().sum().sum(), '->', ventas_limpias.isna().sum().sum())
print('Filas duplicadas         :', ventas.duplicated().sum(), '->', ventas_limpias.duplicated().sum())
print('Ciudades distintas       :', ventas['ciudad'].nunique(), '->', ventas_limpias['ciudad'].nunique())
print('Precio maximo            :', f"{ventas['precio'].max():,.0f}", '->', f"{ventas_limpias['precio'].max():,.0f}")
print('Calificacion maxima      :', ventas['calificacion'].max(), '->', ventas_limpias['calificacion'].max())
print('-' * 42)
print('Media de precio          :', f"{ventas['precio'].mean():,.0f}", '->', f"{ventas_limpias['precio'].mean():,.0f}")

---
## Parte 9 - Formatos: CSV, JSON y HDF5

| Formato | Guardar | Leer | Para que sirve |
|---|---|---|---|
| **CSV** | `to_csv()` | `read_csv()` | texto plano, estandar universal |
| **JSON** | `to_json()` | `read_json()` | el idioma de las APIs web |
| **HDF5** | `to_hdf()` | `read_hdf()` | binario y comprimido, para tablas grandes |

HDF5 necesita la libreria `tables` instalada aparte. El `try / except` de abajo intenta
hacer algo y, si falla, atrapa el error y sigue en vez de tumbar el notebook.

In [ ]:
ventas_limpias.to_csv('ventas_limpias.csv', index=False)
desde_csv = pd.read_csv('ventas_limpias.csv')
print('CSV  : guardado y leido ->', desde_csv.shape)

ventas_limpias.to_json('ventas_limpias.json', orient='records', indent=2)
desde_json = pd.read_json('ventas_limpias.json')
print('JSON : guardado y leido ->', desde_json.shape)

with open('ventas_limpias.json') as archivo:
    print('\nPrimeras lineas del JSON:')
    for numero, linea in enumerate(archivo):
        if numero >= 8:
            break
        print('  ' + linea.rstrip())

try:
    ventas_limpias.to_hdf('ventas_limpias.h5', key='ventas', mode='w')
    desde_hdf = pd.read_hdf('ventas_limpias.h5', key='ventas')
    print('\nHDF5 : guardado y leido ->', desde_hdf.shape)
except ImportError:
    print('\nHDF5 : no disponible en este entorno.')
    print('       Falta la libreria PyTables. Se instala con:  pip install tables')

> **Nota sobre Polars:** existe otra libreria de tablas, `polars`, mucho mas rapida que
> Pandas con datos grandes. No entra en este modulo porque la Ruta trabaja con Pandas;
> queda como lectura opcional en el notebook de **Complementos**.

---
---
## TAREA - antes del martes 22 de septiembre

**Esta tarea es la Actividad 2 (25%).** Se entrega `.ipynb` + PDF.

| # | Requisito | Donde lo vimos hoy |
|---|---|---|
| 1 | Cargar un dataset publico en CSV con Pandas | Parte 1.1 |
| 2 | Imputar faltantes con media, mediana o KNN, **y justificar** | Parte 3 |
| 3 | Detectar y tratar duplicados y valores fuera de rango | Partes 4 y 5 |
| 4 | Detectar outliers con Z-Score o IQR | Parte 6 |
| 5 | Escalar una variable numerica con MinMax o Standard | Parte 7 |
| 6 | **Una funcion reutilizable por cada paso** | Parte 8 |
| 7 | Codigo comentado y estructurado | todo el notebook |

Busca un CSV publico con al menos 200 filas, 5 columnas y defectos reales (Kaggle,
datos.gov.co, UCI). Un dataset ya limpio no sirve.

In [ ]:
# TODO: completa el esqueleto con TU dataset.
#
# import pandas as pd
# import numpy as np
# from sklearn.preprocessing import MinMaxScaler, StandardScaler
#
# 1. CARGA
# df = pd.read_csv('tu_archivo.csv')
#
# 2. DIAGNOSTICO: shape, dtypes, isna().sum(), duplicated().sum(), describe()
#
# 3. FUNCIONES (una por paso, con docstring y comentarios):
#      def limpiar_duplicados(df): ...
#      def limpiar_texto(df, columnas): ...
#      def corregir_fuera_de_rango(df, col, minimo, maximo): ...
#      def detectar_outliers_zscore(df, col, umbral=3): ...   <- nueva, escribela tu
#      def quitar_outliers_iqr(df, col, factor=1.5): ...
#      def imputar_faltantes(df, col, estrategia='mediana'): ...
#      def escalar(df, col, metodo='minmax'): ...
#
# 4. PIPELINE: def pipeline_limpieza(df): que las llame en orden y devuelva la tabla.
#
# 5. AUDITORIA: antes y despues (filas, huecos, duplicados, maximos).
#
# 6. JUSTIFICACION escrita en celdas de texto: por que esa imputacion, por que ese
#    metodo de outliers, por que ese escalador. Esto es lo que mas puntos da.

# Escribe tu solucion desde aqui:


### Parte 2 - Responde

1. Por que la mediana aguanta mejor un outlier que la media? Explicalo con un ejemplo tuyo.
2. Que hace `KNNImputer` que no hacen `fillna(media)` ni `fillna(mediana)`? Que exige a cambio?
3. Un dato marcado como outlier, se borra siempre? Da un caso donde borrarlo seria un error.
4. Cual es la diferencia entre `MinMaxScaler` y `StandardScaler`? Cuando da igual usar uno u otro?
5. Por que `df.drop_duplicates()` sin reasignar no cambia nada? Que error habrias visto? Ninguno: explica por que eso lo hace mas peligroso.
6. Por que la rubrica insiste tanto en funciones reutilizables en vez de celdas sueltas?

**Tus respuestas:**

1.
2.
3.
4.
5.
6.

---
## Resumen

| Problema | Detectar | Arreglar |
|---|---|---|
| Faltantes | `df.isna().sum()` | `fillna(media/mediana)`, `KNNImputer`, `dropna()` |
| Duplicados | `df.duplicated().sum()` | `df = df.drop_duplicates()` |
| Texto sucio | `df['col'].unique()` | `.str.strip().str.title()` |
| Fuera de rango | `df[df['col'] > maximo]` | `.clip(min, max)` o volverlo `NaN` e imputar |
| Outliers | Z-Score `abs(z) > 3` / IQR `1.5 * IQR` | eliminar, recortar o justificar |
| Escalas distintas | `df.describe()` | `MinMaxScaler` / `StandardScaler` |

### Las decisiones que hay que saber defender

| Pregunta | Respuesta corta |
|---|---|
| Media o mediana? | mediana si hay outliers; media si la columna es simetrica |
| Z-Score o IQR? | IQR con muestras pequenas o distribuciones torcidas; Z-Score con muestras grandes y simetricas |
| MinMax o Standard? | MinMax si necesitas rango fijo 0-1; Standard si el metodo asume datos centrados |
| Borrar o imputar? | imputar si los datos son pocos; borrar solo si los huecos son marginales |
| Celdas sueltas o funciones? | funciones, siempre. Es criterio de la rubrica |

### El orden correcto

`duplicados -> texto -> fuera de rango -> outliers -> faltantes -> escalar`

Primero se sacan los valores absurdos, **despues** se calcula el promedio con el que se
rellena. Al reves, el absurdo se cuela en el relleno.

**Proxima sesion:** Matplotlib y Seaborn - contar historias con graficos.

*Dudas: consulta al profesor*

---
---
## Ejercicios de refuerzo (opcionales)

Practica por tu cuenta. Van de mas facil a mas retador. Intenta primero sin mirar la pista.

### Ejercicio 1 - Facil
Sobre la tabla `ventas` original, cuenta cuantos huecos hay en total y muestra que
porcentaje del total de celdas representan.

> *Pista: `ventas.isna().sum().sum()` cuenta todos los huecos; el total de celdas es
> `ventas.shape[0] * ventas.shape[1]`.*

In [ ]:
# Tu solucion aqui


### Ejercicio 2 - Medio
Escribe una funcion `detectar_outliers_zscore(df, col, umbral=3)` que devuelva las filas
cuyo Z-Score supere el umbral. Pruebala sobre `ventas` con la columna `precio` y despues
con umbral 2.

> *Pista: calcula `z = (df[col] - df[col].mean()) / df[col].std()` y filtra con
> `df[z.abs() > umbral]`.*

In [ ]:
# Tu solucion aqui


### Ejercicio 3 - Reta
Agrega a `pipeline_limpieza` un parametro `metodo_imputacion='mediana'` que se pase a
`imputar_faltantes`, y otro `metodo_escala='minmax'` que se pase a `escalar`. Corre el
pipeline dos veces sobre `ventas` con combinaciones distintas y compara las medias de
`precio` resultantes.

> *Pista: los parametros del pipeline se reenvian a las funciones internas. Guarda cada
> resultado en una variable distinta para poder compararlos.*

In [ ]:
# Tu solucion aqui
